# Customer Churn Analysis
Exploratory analysis of the churn dataset. Run the cells top to bottom (from the project root or the `notebooks/` folder).

**Churn definition:** a customer is *churned* if `churn_records` has a row with `churn_status = 'Churned'`.
The bundled data is **synthetic**, so treat the findings as a demonstration of the method.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.database import load_tables
from src.features import build_features
from src.analysis import churn_rate_by, age_group, kpis, monthly_churn

tables, source = load_tables()
features = build_features(tables)
print("Source:", source)
features.head()

## 1. Headline numbers

In [ ]:
kpis(features, tables["churn_records"])

## 2. Churn by plan, city and age group

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
churn_rate_by(features, "plan_name").plot.bar(x="plan_name", y="churn_rate_pct", ax=axes[0], legend=False, title="By plan")
churn_rate_by(features, "city").plot.bar(x="city", y="churn_rate_pct", ax=axes[1], legend=False, title="By city")
seg = features.assign(age_group=age_group(features.age))
churn_rate_by(seg, "age_group").sort_values("age_group").plot.bar(x="age_group", y="churn_rate_pct", ax=axes[2], legend=False, title="By age group")
for ax in axes: ax.set_ylabel("churn rate (%)"); ax.set_xlabel("")
plt.tight_layout(); plt.show()
churn_rate_by(features, "city")

Small groups swing a lot. Check the `customers` column before reading anything into a single bar.

## 3. Behaviour: tickets and failed payments

In [ ]:
features["segment"] = features.churned.map({1: "Churned", 0: "Retained"})
features.groupby("segment")[["total_tickets", "unresolved_tickets", "failed_payment_rate", "tenure_days"]].mean().round(2)

In [ ]:
bands = pd.cut(features.failed_payment_rate, [-0.001, 0, 0.1, 0.2, 1], labels=["0%", "<10%", "10-20%", "20%+"])
features.groupby(bands, observed=True).churned.agg(customers="size", churn_rate="mean").round(3)

## 4. Churn over time and reasons

In [ ]:
monthly_churn(tables["churn_records"]).plot(figsize=(10, 3), title="Churn events per month")
plt.show()
tables["churn_records"].query("churn_status == 'Churned'").churn_reason.value_counts()

## 5. Model results
Produced by `python src/train_model.py`. Features exclude tenure and payment counts because they leak the outcome
(they stop growing once a customer churns).

In [ ]:
import json
meta = json.loads((ROOT / "models" / "metrics.json").read_text())
print("Best model:", meta["best_model"])
pd.DataFrame(meta["results"]).T

In [ ]:
pd.read_csv(ROOT / "reports" / "feature_importance.csv")

In [ ]:
pd.read_csv(ROOT / "data" / "churn_risk_scores.csv").query("risk_band == 'High'").head(15)